# Merkle Trees
This notebook is a compantion to the medium article [Inside OpenCerts: How Merkle Trees Verify Singapore’s Digital Diplomas](https://medium.com/mitb-for-all/inside-opencerts-how-merkle-trees-verify-singapores-digital-diplomas-61f60d13700e)


## The problem

Say University A graduates 8,000 students in one cohort. Putting all 8,000 diploma
records directly on-chain would be slow, expensive, and would leak every
graduate's private academic data to anyone who reads the chain.

University A could instead hash the whole batch into a single fingerprint and
publish *that*. But a flat hash of "all 8,000 diplomas concatenated
together" has a problem: to prove *your* diploma is part of that batch,
you'd need to hand over all 8,000 diplomas so the verifier can re-hash
everything and check it matches. That defeats the purpose -- no privacy,
and no efficiency gain.

## The Merkle tree solution

A **Merkle tree** solves this by hashing documents in pairs, repeatedly,
until only one hash is left -- the **Merkle root**:

```
        Root
       /    \
     H01     H23
    /  \    /  \
  H0   H1  H2   H3
   |    |   |    |
  D0   D1  D2   D3
```

- `D0..D3` are the individual diplomas (the **leaves**).
- Only the **root** ever needs to be published (e.g. as a blockchain
  transaction) -- a single, fixed-size fingerprint of the *entire*
  batch, no matter how large the batch is.
- Any single graduate can prove their diploma is part of that batch
  using only a handful of extra hashes -- a **Merkle proof** -- instead
  of the whole batch. For N documents, a proof needs only `log2(N)`
  hashes.
- Anyone can verify the proof by recomputing hashes up the tree and
  checking the result matches the published root -- without ever
  seeing any other graduate's diploma.

This is the concept behind how Singapore Universities could publish ONE
hash for a whole batch of diplomas, while any one graduate proves their
specific diploma was in that batch -- without needing, revealing, or
even seeing anyone else's diploma.

> All using blockchain.

In [1]:
import hashlib


def sha256(data: str) -> str:
    """Deterministic hash helper used throughout."""
    return hashlib.sha256(data.encode()).hexdigest()

In [2]:
class MerkleTree:
    """A binary tree of hashes, built bottom-up from a list of documents.

    Each document becomes a "leaf" (individually hashed). Leaves are
    paired up and hashed together to form the next layer up. This
    repeats until exactly one hash remains -- the "Merkle root" -- which
    is the ONLY thing that needs to be published on-chain.

    Caveats / assumptions:
      - Relies on SHA-256 being collision-resistant: this whole scheme
        breaks if two different documents can be found that hash the
        same.
      - Leaf order matters and must be agreed/fixed ahead of time --
        the tree commits to a specific ORDERED batch, not just a set.
      - Odd-sized layers duplicate the last hash so it can pair with
        itself (see `_build`). This is the simplest fix for "odd number
        of nodes", but it's not free: naive duplicate-padding is what
        let an attacker on early Bitcoin construct a modified
        transaction list with the SAME Merkle root as the original, by
        duplicating the last transaction (CVE-2012-2459). Production
        systems (e.g. Certificate Transparency, RFC 6962) instead use
        domain-separated hashing -- leaves and internal nodes are
        hashed with different prefixes -- to remove this ambiguity.
        This notebook keeps the simple version for teaching purposes.
      - A Merkle tree proves a document was included in the batch that
        produced a given root -- it says nothing about whether the
        document's CONTENTS were true or correct at issuance. Garbage
        in, verifiably-tamper-evident garbage out.

    Attributes:
        leaves: Hashes of the original documents, in order.
        layers: Every layer of the tree, from leaves (layers[0]) up to
            the root (layers[-1], containing exactly one hash).
    """

    def __init__(self, documents: list[str]) -> None:
        if not documents:
            raise ValueError("Need at least one document to build a tree.")
        self.leaves: list[str] = [sha256(doc) for doc in documents]
        self.layers: list[list[str]] = [self.leaves]
        self._build()

    def _build(self) -> None:
        """Repeatedly hash pairs of the current layer until one hash remains."""
        current = self.layers[0]
        while len(current) > 1:
            next_layer: list[str] = []
            for i in range(0, len(current), 2):
                left = current[i]
                # odd count -> duplicate the last item so it can pair with itself
                right = current[i + 1] if i + 1 < len(current) else current[i]
                next_layer.append(sha256(left + right))
            self.layers.append(next_layer)
            current = next_layer

    @property
    def root(self) -> str:
        """The single hash summarizing the ENTIRE batch of documents."""
        return self.layers[-1][0]

    def get_proof(self, leaf_index: int) -> list[tuple[str, str]]:
        """Build the minimal sibling-hash path proving a document's membership.

        This is the entire point of the structure: instead of needing
        ALL other documents, you need exactly one sibling hash per
        LEVEL of the tree -- log2(N) hashes total, not N-1.

        Args:
            leaf_index: Position of the document to prove membership for.

        Returns:
            A list of (position, hash) pairs. Position ("left"/"right")
            tells the verifier which side of the combination the
            sibling hash belongs on.
        """
        proof: list[tuple[str, str]] = []
        index = leaf_index
        for layer in self.layers[:-1]:  # every layer except the root itself
            is_right_node = index % 2 == 1
            pair_index = index - 1 if is_right_node else index + 1
            if pair_index < len(layer):
                sibling_hash = layer[pair_index]
            else:
                # `index` is the odd node out at the end of this layer --
                # `_build` paired it with itself, so the "sibling" IS
                # this node's own hash. It must still be included, or
                # the verifier's recomputed hash won't match the real
                # root for this leaf (this case is easy to miss because
                # it never triggers when the batch size is a power of 2
                # -- see the demos below, which use 8 and 4 documents).
                sibling_hash = layer[index]
            position = "left" if is_right_node else "right"
            proof.append((position, sibling_hash))
            index //= 2
        return proof

In [3]:
def verify_proof(leaf_hash: str, proof: list[tuple[str, str]], root: str) -> bool:
    """Independently recompute the root from a leaf hash + its proof, and
    check it matches the published root -- without ever seeing any other
    document in the batch.

    Args:
        leaf_hash: Hash of the document being proven.
        proof: Sibling hashes returned by MerkleTree.get_proof().
        root: The published root hash to check against.

    Returns:
        True if the leaf genuinely belongs to the tree that produced root.
    """
    current_hash = leaf_hash
    for position, sibling_hash in proof:
        if position == "left":
            current_hash = sha256(sibling_hash + current_hash)
        else:
            current_hash = sha256(current_hash + sibling_hash)
    return current_hash == root

In [4]:
print("=== University A issues 8 diplomas in one graduation batch ===\n")
diplomas = [
    "Diploma: Alice Tan, Computer Science, 2026",
    "Diploma: Bob Lim, Chemical Engineering, 2026",
    "Diploma: Carol Ng, Economics, 2026",
    "Diploma: Titus Lim, MITB AI, 2026",
    "Diploma: Emma Koh, Mathematics, 2026",
    "Diploma: Farid Rahman, Physics, 2026",
    "Diploma: Grace Wong, Biology, 2026",
    "Diploma: Hafiz Ismail, Statistics, 2026",
]
tree = MerkleTree(diplomas)
print(f"Merkle root (the ONLY thing University A publishes on-chain):\n  {tree.root}\n")

print("=== Titus wants to prove HIS diploma is in the batch ===\n")
titus_index = diplomas.index("Diploma: Titus Lim, MITB AI, 2026")
proof = tree.get_proof(titus_index)
print(f"  Titus's diploma is at position {titus_index} of {len(diplomas)}.")
print(f"  His proof needs only {len(proof)} sibling hashes "
        f"(not the other {len(diplomas) - 1} diplomas!):")
for pos, h in proof:
    print(f"    combine on the {pos}: {h[:16]}...")

=== University A issues 8 diplomas in one graduation batch ===

Merkle root (the ONLY thing University A publishes on-chain):
  f97175862043d258e0a467d6e777832d375b30d018ad8b1985e52eb6467c0782

=== Titus wants to prove HIS diploma is in the batch ===

  Titus's diploma is at position 3 of 8.
  His proof needs only 3 sibling hashes (not the other 7 diplomas!):
    combine on the left: 4b2493046335dc12...
    combine on the left: 4374db001d70618f...
    combine on the right: bdda8da6694c5eeb...


In [5]:
print("\n=== Employer verifies, using ONLY Titus's document + his proof + the public root ===\n")
titus_leaf_hash = sha256("Diploma: Titus Lim, MITB AI, 2026")
is_valid = verify_proof(titus_leaf_hash, proof, tree.root)
print(f"  Verified genuine? {is_valid}")

print("\n=== What if Titus tries to upgrade his own grade after the fact? ===\n")
forged_leaf_hash = sha256("Diploma: Titus Lim, MITB AI, 2026, FIRST CLASS HONOURS")
is_valid_forged = verify_proof(forged_leaf_hash, proof, tree.root)
print(f"  Verified genuine? {is_valid_forged}")

print("\n=== Did any of this require touching Alice, Bob, Carol, or anyone else's diploma? ===\n")
print("  Not once. That's the entire point.")


=== Employer verifies, using ONLY Titus's document + his proof + the public root ===

  Verified genuine? True

=== What if Titus tries to upgrade his own grade after the fact? ===

  Verified genuine? False

=== Did any of this require touching Alice, Bob, Carol, or anyone else's diploma? ===

  Not once. That's the entire point.


# The full Merkle certificate lifecycle

This wires together everything into ONE coherent story instead
of separate demos, so you can see how Layer 1 (Merkle trees) and
Layer 2 (the PoS blockchain) actually interlock:

  1. University A builds a Merkle tree of a diploma batch (MerkleTree)
  2. University A submits ONLY the resulting root as a transaction's payload
  3. A Proof-of-Stake validator wins the weighted lottery, includes that
     transaction in a block, and earns a reward for doing so
     (Validator, Block, Blockchain, pick_proposer)
  4. Years later, an employer verifies one graduate's certificate using
     ONLY that graduate's own file + the root now permanently on-chain
     (Certificate, issue_batch, verify_certificate)

In [6]:
import copy
import json
import secrets
import time
from dataclasses import dataclass

In [7]:
@dataclass
class Certificate:
    """What a graduate actually keeps -- self-contained and independently
    verifiable forever, no tree required after issuance."""
    holder: str
    content: str
    proof: list[tuple[str, str]]

In [8]:
def issue_batch(holders: list[str], contents: list[str]) -> tuple[dict[str, Certificate], str]:
    """University A's one-time issuance: build the tree, hand out proofs, let the
    tree die. Returns (certificates, root-to-be-published-on-chain)."""
    tree = MerkleTree(contents)
    root = tree.root
    certificates = {
        holder: Certificate(holder=holder, content=content, proof=tree.get_proof(i))
        for i, (holder, content) in enumerate(zip(holders, contents))
    }
    return certificates, root
    # `tree` dies here -- nothing below this line ever references it again.


def verify_certificate(cert: Certificate, published_root: str) -> bool:
    """Standalone verification: only this certificate + an on-chain root."""
    return verify_proof(sha256(cert.content), cert.proof, published_root)

## From Merkle roots to a blockchain: bringing in Proof of Stake

Everything above (`MerkleTree`, `verify_proof`, `Certificate`,
`issue_batch`) is **Layer 1**: a way to commit to a whole batch of
documents with a single hash, and prove membership cheaply. On its own
it's just a data structure -- it doesn't give you tamper-evidence over
TIME, or agreement between many parties about which root is "the real
one".

That's what a blockchain adds. The classes below (`Validator`, `Block`,
`pick_proposer`, `Blockchain`) are **Layer 2**: a Proof-of-Stake (PoS)
blockchain, covered in depth in `2. blockchain_pos.ipynb` (stake-weighted
validator selection, slashing for equivocation, historical-eligibility
checks). Here they exist only so a Merkle root has somewhere permanent
and tamper-evident to live: University A's diploma-batch root becomes the `data`
field of one block, instead of a floating hash nobody can pin down.

In [9]:
BLOCK_REWARD: float = 2.0
FEE_PER_TX: float = 0.1


class Validator:
    """A staked network participant, eligible to propose blocks."""

    def __init__(self, name: str, stake: float) -> None:
        self.name = name
        self.stake = stake
        self.slashed_at_height: int | None = None

    @property
    def is_slashed(self) -> bool:
        return self.slashed_at_height is not None

    def was_eligible_at(self, height: int) -> bool:
        if self.slashed_at_height is None:
            return True
        return height < self.slashed_at_height

    def __repr__(self) -> str:
        status = f"SLASHED at height {self.slashed_at_height}" if self.is_slashed else "active"
        return f"{self.name}: stake={self.stake:.2f} [{status}]"

In [10]:
class Block:
    """A chain entry. `data` here will typically hold a Merkle root, NOT
    raw content -- that's the whole point of Layer 1 above."""

    def __init__(
        self, index: int, data: str, previous_hash: str, proposer: str, num_transactions: int = 1
    ) -> None:
        self.index = index
        self.timestamp = time.time()
        self.data = data
        self.previous_hash = previous_hash
        self.proposer = proposer
        self.num_transactions = num_transactions
        self.hash = self.compute_hash()

    def compute_hash(self) -> str:
        block_contents = json.dumps(
            {
                "index": self.index,
                "timestamp": self.timestamp,
                "data": self.data,
                "previous_hash": self.previous_hash,
                "proposer": self.proposer,
                "num_transactions": self.num_transactions,
            },
            sort_keys=True,
        )
        return hashlib.sha256(block_contents.encode()).hexdigest()

    def __repr__(self) -> str:
        return (
            f"Block #{self.index} proposed by {self.proposer} ({self.num_transactions} txs)\n"
            f"  data (Merkle root): {self.data[:24]}...\n"
        )


def pick_proposer(validators: list[Validator]) -> Validator:
    """Cryptographically secure, stake-weighted proposer selection.
    See `2. blockchain_pos.ipynb` for the full rationale on `secrets` vs
    `random`: a predictable PRNG would let an attacker forecast (or try
    to bias, i.e. "grind") who proposes next."""
    active = [v for v in validators if not v.is_slashed]
    weights = [v.stake for v in active]
    total = sum(weights)
    pick = secrets.randbelow(int(total * 1000)) / 1000
    cumulative = 0.0
    for validator, weight in zip(active, weights):
        cumulative += weight
        if pick < cumulative:
            return validator
    return active[-1]


def slash(validator: Validator, height: int, penalty_fraction: float = 1.0) -> None:
    lost = validator.stake * penalty_fraction
    validator.stake -= lost
    validator.slashed_at_height = height
    print(f"  SLASHED: {validator.name} loses {lost:.2f} coins at height {height}.")


class Blockchain:
    """Owns the chain; PoS-flavoured legitimacy rules. See
    `2. blockchain_pos.ipynb` for the historical-eligibility rule this
    already includes: a slashed validator's PAST blocks, proposed while
    still honest, remain valid -- only blocks proposed at or after their
    slash height are rejected."""

    def __init__(self, validators: list[Validator]) -> None:
        self.validators = validators
        self.chain: list[Block] = [Block(0, "Genesis Block", "0" * 64, "network")]

    def add_block(self, data: str, num_transactions: int = 1) -> tuple[Block, Validator]:
        proposer = pick_proposer(self.validators)
        previous_block = self.chain[-1]
        new_block = Block(len(self.chain), data, previous_block.hash, proposer.name, num_transactions)
        self.chain.append(new_block)
        proposer.stake += BLOCK_REWARD + num_transactions * FEE_PER_TX
        return new_block, proposer

    def is_valid(self) -> tuple[bool, str]:
        validators_by_name = {v.name: v for v in self.validators}
        for i in range(1, len(self.chain)):
            current, previous = self.chain[i], self.chain[i - 1]
            if current.hash != current.compute_hash():
                return False, f"Block #{current.index} was tampered with directly."
            if current.previous_hash != previous.hash:
                return False, f"Block #{current.index} is disconnected from Block #{previous.index}."
            proposer = validators_by_name.get(current.proposer)
            if proposer is None:
                return False, f"Block #{current.index} was proposed by an unknown validator."
            if not proposer.was_eligible_at(current.index):
                return False, f"Block #{current.index} was proposed by an already-slashed validator."
        return True, "Chain is valid."

In [11]:
print("=== Step 1: University A builds the Merkle tree for a diploma batch ===\n")
holders = ["Alice Tan", "Bob Lim", "Carol Ng", "Titus Lim"]
majors = ["Computer Science", "Chemical Engineering", "Economics", "AI Engineering"]
contents = [f"Diploma: {h}, {m}, 2026" for h, m in zip(holders, majors)]

certificates, merkle_root = issue_batch(holders, contents)
print(f"  Merkle root of this 4-diploma batch: {merkle_root[:24]}...")
print(f"  {len(certificates)} Certificate objects handed out, tree already gone.\n")

=== Step 1: University A builds the Merkle tree for a diploma batch ===

  Merkle root of this 4-diploma batch: f0c1f18b7c16292e7fda4d25...
  4 Certificate objects handed out, tree already gone.



In [12]:
print("=== Step 2: the root gets submitted to a PoS blockchain as a transaction ===\n")
validators = [
    Validator("Acme-Node", stake=500),
    Validator("Guy-A-Node", stake=100),
    Validator("SketchyGuy-Node", stake=50),
]
chain = Blockchain(validators)

# a few unrelated prior blocks, so this isn't suspiciously the first thing ever
chain.add_block("Some unrelated earlier transaction batch", num_transactions=5)
chain.add_block("Another unrelated batch", num_transactions=3)

diploma_block, proposer = chain.add_block(
    data=merkle_root, num_transactions=len(certificates)
)
print(f"  {proposer.name} won the lottery for this slot and included University A's transaction.")
print(f"  {proposer.name} earned {BLOCK_REWARD + len(certificates) * FEE_PER_TX:.2f} coins for it.")
print(f"  {diploma_block}")

=== Step 2: the root gets submitted to a PoS blockchain as a transaction ===

  Guy-A-Node won the lottery for this slot and included University A's transaction.
  Guy-A-Node earned 2.40 coins for it.
  Block #3 proposed by Guy-A-Node (4 txs)
  data (Merkle root): f0c1f18b7c16292e7fda4d25...



In [13]:
valid, msg = chain.is_valid()
print(f"  Chain valid? {valid} -- {msg}\n")

  Chain valid? True -- Chain is valid.



In [14]:
print("=== Step 3: years later, an employer verifies Titus's certificate ===\n")
titus_cert = certificates["Titus Lim"]

# The employer doesn't need "the tree" or "University A's database" -- just the
# root that's now permanently sitting inside diploma_block on-chain.
published_root = diploma_block.data
is_valid = verify_certificate(titus_cert, published_root)
print(f"  Looked up block #{diploma_block.index} on-chain, read its root.")
print("  Verified using ONLY Titus's own certificate + that root.")
print(f"  Verified genuine? {is_valid}\n")

=== Step 3: years later, an employer verifies Titus's certificate ===

  Looked up block #3 on-chain, read its root.
  Verified using ONLY Titus's own certificate + that root.
  Verified genuine? True



In [15]:
print("=== Step 4: what if Titus's certificate is tampered with? ===\n")
forged = copy.deepcopy(titus_cert)
forged.content += ", FIRST CLASS HONOURS"
is_valid_forged = verify_certificate(forged, published_root)
print(f"  Tampered content: {forged.content}")
print(f"  Verified genuine? {is_valid_forged}\n")

print("=== Final validator stakes (notice who earned from this) ===")
for v in validators:
    print(f"  {v}")

=== Step 4: what if Titus's certificate is tampered with? ===

  Tampered content: Diploma: Titus Lim, AI Engineering, 2026, FIRST CLASS HONOURS
  Verified genuine? False

=== Final validator stakes (notice who earned from this) ===
  Acme-Node: stake=504.80 [active]
  Guy-A-Node: stake=102.40 [active]
  SketchyGuy-Node: stake=50.00 [active]
